In [24]:
import sys
sys.path.append("..")          # lets the notebook find files in src/
 
import numpy as np
import pandas as pd
from src.db import get_engine
 
engine = get_engine()


In [25]:
rng = np.random.default_rng(42)        # 42 is a fixed seed: everyone gets identical data
states = ["Selangor", "Johor", "Penang", "Sabah", "Sarawak", "Kelantan"]
categories = ["Fiction", "Business", "Technology", "Children"]
n = 500
 
sales = pd.DataFrame({
    "sale_date": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 90, n), unit="D"),
    "state": rng.choice(states,n),
    "category": rng.choice(categories, n),
    "amount_myr": rng.gamma(2.0, 40.0, n).round(2),
})
targets = pd.DataFrame({"state": states, "q1_target_myr": [9000, 6000, 5000, 3000, 3500, 2500]})
sales.head()

,sale_date,state,category,amount_myr
0,2026-01-09,Selangor,Children,61.82
1,2026-03-11,Sabah,Technology,4.63
2,2026-02-28,Kelantan,Technology,29.46
3,2026-02-09,Kelantan,Children,82.35
4,2026-02-08,Johor,Fiction,23.42


In [26]:
sales.to_sql("practice_sales", engine, if_exists="replace", index=False)
targets.to_sql("practice_targets", engine, if_exists="replace", index=False)
print("Loaded", len(sales), "sales rows and", len(targets), "targets")


Loaded 500 sales rows and 6 targets


In [27]:
def q(sql):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql(sql, engine)
q("SELECT COUNT(*) AS row_count FROM practice_sales")

,row_count
0,500


In [28]:
q("""
SELECT state, COUNT(*) AS orders, ROUND(SUM(amount_myr)::numeric, 2) AS revenue
FROM practice_sales
GROUP BY state
ORDER BY revenue DESC
""")


,state,orders,revenue
0,Selangor,93,7732.62
1,Sabah,96,7341.15
2,Kelantan,86,6441.66
3,Sarawak,83,6411.48
4,Johor,72,5920.78
5,Penang,70,4986.58


In [29]:
q("""
SELECT s.state,
       ROUND(SUM(s.amount_myr)::numeric, 2) AS revenue,
       t.q1_target_myr,
       ROUND((SUM(s.amount_myr) / t.q1_target_myr * 100)::numeric, 1) AS pct_of_target
FROM practice_sales s
JOIN practice_targets t ON s.state = t.state
GROUP BY s.state, t.q1_target_myr
ORDER BY pct_of_target DESC
""")


,state,revenue,q1_target_myr,pct_of_target
0,Kelantan,6441.66,2500,257.7
1,Sabah,7341.15,3000,244.7
2,Sarawak,6411.48,3500,183.2
3,Penang,4986.58,5000,99.7
4,Johor,5920.78,6000,98.7
5,Selangor,7732.62,9000,85.9


In [30]:
q("""
WITH state_rev AS (
    SELECT state, SUM(amount_myr) AS revenue
    FROM practice_sales
    GROUP BY state

)
SELECT state, ROUND(revenue::numeric, 2) AS revenue
FROM state_rev
WHERE revenue > (SELECT AVG(revenue) FROM state_rev)
""")


,state,revenue
0,Sabah,7341.15
1,Selangor,7732.62


In [32]:
q("""
WITH daily AS (
    SELECT sale_date::date AS day, SUM(amount_myr) AS revenue
    FROM practice_sales
    GROUP BY 1
)
SELECT day,
     ROUND(revenue::numeric, 2)AS revenue,
     ROUND(SUM(revenue) OVER (ORDER BY day)::numeric, 2) AS running_total

FROM daily
ORDER BY day
""")


,day,revenue,running_total
0,2026-01-01,196.26,196.26
1,2026-01-02,531.09,727.35
2,2026-01-03,362.22,1089.57
3,2026-01-04,359.41,1448.98
4,2026-01-05,133.17,1582.15
...,...,...,...
85,2026-03-27,246.91,36984.01
86,2026-03-28,468.44,37452.45
87,2026-03-29,866.08,38318.53
88,2026-03-30,66.82,38385.35


In [33]:
q("""
SELECT state, category,
       ROUND(SUM(amount_myr)::numeric, 2) AS revenue,
       RANK() OVER (PARTITION BY state ORDER BY SUM(amount_myr) DESC) AS rank_in_state
FROM practice_sales
GROUP BY state, category
ORDER BY state, rank_in_state
""")


,state,category,revenue,rank_in_state
0,Johor,Technology,1944.04,1
1,Johor,Business,1907.24,2
2,Johor,Children,1085.64,3
3,Johor,Fiction,983.86,4
4,Kelantan,Business,1860.03,1
5,Kelantan,Children,1841.46,2
6,Kelantan,Fiction,1420.00,3
7,Kelantan,Technology,1320.17,4
8,Penang,Fiction,1595.48,1
9,Penang,Business,1495.85,2
